# Exercise 08 — Text Generation with Hugging Face

In week 6 you sampled text from your mini GPT, and you called GPT-2's `generate` once with greedy decoding and once with nucleus sampling. Last week you used `generate` to let Qwen solve math problems. This week is about that one method: how a language model, which only ever predicts *the next token*, is turned into something that writes whole texts, and how the choices made while doing so change what comes out.

This week has three notebooks:

- **The `generate` API** (this notebook): the knobs that Hugging Face gives you, and what `generate` actually returns.
- **Decoding strategies** (notebook 1): greedy search, beam search and sampling with temperature, top-k and top-p, compared with numbers and not only by eye.
- **Decoding from scratch** (notebook 2): you implement these strategies yourself, check them against Hugging Face, and plug your own rules into `generate`.

This notebook is less about writing algorithms and more about getting to know the tool. Most tasks are short; take the time to play with the settings.

## What you will do
1. **`generate`**: what goes in, what comes out, and how to get only the new text.
2. **`GenerationConfig`, length and stopping**: when does generation end?
3. **Shaping the output**: repetition penalties, n-gram blocking and banned words.
4. **Looking inside**: the scores of every step, token probabilities, and what the logits processors do.
5. **Batched generation**: why decoder-only models need *left* padding.
6. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook downloads GPT-2** (~550 MB) unless it is still cached from week 6. Everything runs on a CPU in a few seconds per cell.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, set_seed

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

## 1. `generate`: what goes in and what comes out

We use **GPT-2** (124 M parameters), the same model as in week 6. It is small enough for a CPU and, being a plain pretrained model, it shows the strengths and the failure modes of each decoding method very clearly.

GPT-2 has no padding token. `generate` needs one (for example to fill up finished sequences in a batch), so we tell it to use the end-of-text token `<|endoftext|>` for padding. Setting it once in the model's generation config saves us from passing `pad_token_id=...` to every call, and from a warning each time.

In [ ]:
MODEL_NAME = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

model.generation_config.pad_token_id = tokenizer.eos_token_id
print(f"{sum(p.numel() for p in model.parameters()):,} parameters, vocabulary of {len(tokenizer):,} tokens")

`generate` takes the tokenized prompt (the `input_ids` and the `attention_mask`) and returns a tensor of token ids of shape `(batch_size, prompt_length + new_tokens)`. Note that the output **starts with the prompt**: a decoder-only model simply keeps appending to its input.

In [ ]:
inputs = tokenizer("Once upon a time", return_tensors="pt").to(device)
print("input ids: ", inputs["input_ids"])

outputs = model.generate(**inputs, max_new_tokens=30)
print("output ids:", outputs)
print()
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

**1.1 Complete `continue_text`**, which returns only the *new* text, without the prompt. All keyword arguments are passed on to `generate`, so you can use it with any setting later on.

Hint: the prompt has `inputs["input_ids"].shape[1]` tokens. Slice them off the output *before* decoding. (Slicing the decoded string by `len(prompt)` characters is fragile: decoding does not always give back exactly the characters you put in.)

In [ ]:
def continue_text(prompt, **generate_kwargs):
    # Generate a continuation of the prompt and return only the new text
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = ...    # TODO: call model.generate with the inputs and the generate_kwargs
    new_ids = ...    # TODO: the new tokens of the first (and only) sequence, without the prompt
    return tokenizer.decode(new_ids, skip_special_tokens=True)


print(repr(continue_text("Once upon a time", max_new_tokens=30)))

In [ ]:
# ✅ Check continue_text
text = continue_text("Once upon a time", max_new_tokens=10)
assert isinstance(text, str), "continue_text should return a string"
assert not text.startswith("Once upon a time"), "the prompt should not be part of the returned text"
assert len(tokenizer(text)["input_ids"]) <= 12, "max_new_tokens=10 should give (about) 10 new tokens"
assert text == continue_text("Once upon a time", max_new_tokens=10), "by default generate is deterministic (greedy)"
print("Looks good ✅")

**1.2 Read the continuation of *Once upon a time* above.** What is wrong with it? Without any settings, `generate` uses **greedy decoding**: at every step it takes the single most likely next token. Why would that lead to this kind of text? (You saw the same thing in week 6.)

---

*Your answer here:*

---

## 2. `GenerationConfig`: length and stopping

Every argument you can pass to `generate` is a field of a [`GenerationConfig`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig). The model comes with a default one, which `generate` uses for every argument you do not pass. For GPT-2 it is almost empty, which means greedy decoding:

In [ ]:
print(model.generation_config)

Instead of passing many keyword arguments to every call, you can also collect them in your own `GenerationConfig` and pass it with `generation_config=...`. This is handy when you want to reuse the same settings, or save them together with a model. The two calls below do the same thing:

In [ ]:
config = GenerationConfig(max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

a = model.generate(**inputs, generation_config=config)
b = model.generate(**inputs, max_new_tokens=20, do_sample=False)
print(torch.equal(a, b))

### When does generation stop?

Generation stops as soon as **one** of these happens:

- **`max_new_tokens`** new tokens have been generated (always set it, because a model may never stop on its own),
- the model generates the **end-of-sequence token** (`eos_token_id`). For GPT-2 that is `<|endoftext|>`, which separated documents during pretraining. Chat models have their own end-of-turn tokens,
- one of the **`stop_strings`** appears in the generated text (`generate` then also needs the `tokenizer`, to check strings rather than token ids),
- **`max_time`** seconds have passed.

**`min_new_tokens`** works the other way round: before that many tokens have been generated, the EOS token is forbidden (its probability is set to zero).

**2.1 Complete `first_sentence`**, which generates only the first sentence of a continuation: generation stops at the first `"."`, `"!"` or `"?"`.

In [ ]:
def first_sentence(prompt, max_new_tokens=60):
    # The continuation of the prompt up to and including the first ".", "!" or "?"
    return continue_text(
        prompt,
        max_new_tokens=max_new_tokens,
        # TODO: pass the stop strings, and the tokenizer that generate needs to check them
    )


for prompt in ["The weather in Aarhus is", "My favourite food is", "The meeting was cancelled because"]:
    print(f"{prompt}|{first_sentence(prompt)}")

In [ ]:
# ✅ Check first_sentence
for prompt in ["The weather in Aarhus is", "My favourite food is", "The meeting was cancelled because"]:
    sentence = first_sentence(prompt)
    n_tokens = len(tokenizer(sentence)["input_ids"])
    assert sentence.endswith((".", "!", "?")) or n_tokens >= 58, f"generation should stop at an end of sentence: {sentence!r}. Did you pass the stop strings and the tokenizer?"
print("Looks good ✅")

### The EOS token is just a token

To the model, the end-of-sequence token is a token like any other: generation stops because `generate` *checks* for it after every step. That means you can declare any token to be the EOS token. For example, if you make the full stop `"."` the EOS token, generation stops at the end of the first sentence, as above, but without checking strings.

**2.2 Generate a continuation that is at least 20 tokens long and ends at a full stop.** Use `eos_token_id` and `min_new_tokens`.

Hint: `tokenizer.convert_tokens_to_ids(".")` gives you the id of the full stop.

In [ ]:
period_id = tokenizer.convert_tokens_to_ids(".")
prompt = "The weather in Aarhus is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    # TODO: make the full stop the EOS token, and forbid it during the first 20 new tokens
)
new_ids = outputs[0, inputs["input_ids"].shape[1]:]
print(f"{len(new_ids)} new tokens: {prompt}|{tokenizer.decode(new_ids)}")

In [ ]:
# ✅ Check your settings
ids = new_ids.tolist()
assert period_id not in ids[:19], "the full stop should be forbidden during the first 20 new tokens. Did you set min_new_tokens?"
assert ids[-1] == period_id or len(ids) == 80, "generation should end at the first full stop. Did you set eos_token_id?"
print("Looks good ✅")

**2.3 Stopping in practice.** A chat model is asked a question and should answer it and then stop. How does it know when to stop? And what would happen if you prompted the *base* GPT-2 with `"Question: What is the capital of Denmark?\nAnswer:"` and only set `max_new_tokens=200`?

---

*Your answer here:*

---

## 3. Shaping the output

Some arguments of `generate` do not change *how* the next token is chosen, but *which tokens are allowed* and how likely they are. Before each choice, `generate` passes the logits through a list of **logits processors**, each of which modifies them. Three of them:

- **`repetition_penalty`** (Keskar et al., 2019): makes every token that already appears in the text less likely. A value of 1.0 means no penalty; 1.2–1.3 is common. You implement it yourself in notebook 2.
- **`no_repeat_ngram_size`**: forbids any n-gram (sequence of *n* tokens) that has already been generated. With `no_repeat_ngram_size=3`, no three tokens in a row can occur twice.
- **`bad_words_ids`**: a list of token sequences that may never be generated.

The helper below finds n-grams of token ids that occur more than once.

In [ ]:
def repeated_ngrams(ids, n):
    # The n-grams (as tuples of token ids) that occur more than once in ids
    ngrams = [tuple(ids[i : i + n]) for i in range(len(ids) - n + 1)]
    return {ngram for ngram in ngrams if ngrams.count(ngram) > 1}


def generate_ids(prompt, **generate_kwargs):
    # The new token ids of a continuation of the prompt, as a list
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, **generate_kwargs)
    return outputs[0, inputs["input_ids"].shape[1]:].tolist()


prompt = "I enjoy walking with my cute dog"
greedy_ids = generate_ids(prompt, max_new_tokens=60)
print(tokenizer.decode(greedy_ids))
print(f"\n{len(repeated_ngrams(greedy_ids, 3))} different 3-grams occur more than once")

**3.1 Fix the repetition.** Generate a greedy continuation of the same prompt with a repetition penalty of 1.3 (`rep_ids`), and one in which no 3-gram is repeated (`ngram_ids`).

In [ ]:
rep_ids = ...    # TODO: greedy, 60 new tokens, repetition penalty 1.3
ngram_ids = ...  # TODO: greedy, 60 new tokens, no repeated 3-grams

print("===== repetition_penalty=1.3 =====")
print(tokenizer.decode(rep_ids))
print("\n===== no_repeat_ngram_size=3 =====")
print(tokenizer.decode(ngram_ids))

In [ ]:
# ✅ Check your generations
assert len(rep_ids) == 60 and len(ngram_ids) == 60, "generate 60 new tokens"
assert rep_ids != greedy_ids, "the repetition penalty should change the output"
# the generated 3-grams must not repeat, also not a 3-gram that overlaps the prompt
full = tokenizer(prompt)["input_ids"] + ngram_ids
assert not repeated_ngrams(full, 3), "no 3-gram should occur twice. Did you set no_repeat_ngram_size=3?"
print("Looks good ✅")

**3.2 Look at the two continuations.** Is the repetition gone? Did the texts get *good*? Can you think of a text in which forbidding all repeated 3-grams would be a problem?

---

*Your answer here:*

---

### Banning words

`bad_words_ids` is a list of **token sequences**, not of words. This is where BPE tokenization (week 6) becomes important: GPT-2's tokens include the space *before* a word, so `"dog"` at the start of a text and `" dog"` in the middle of a sentence are different tokens, and so are `" Dog"` and `" dogs"`.

In [ ]:
for word in ["dog", " dog", " Dog", " dogs", " doggy"]:
    ids = tokenizer(word)["input_ids"]
    print(f"{word!r:10} -> {ids} {tokenizer.convert_ids_to_tokens(ids)}")

**3.3 Ban dogs.** Build `banned` for the words `dog`, `dogs`, `Dog` and `Dogs`, each with and without a leading space, and generate a continuation of a prompt about a dog in which none of these token sequences occur.

In [ ]:
words = ["dog", "dogs", "Dog", "Dogs"]
banned = ...  # TODO: a list with the token ids (a list) of every word, with and without a leading space

dogless_ids = generate_ids("I enjoy walking with my cute", max_new_tokens=60, no_repeat_ngram_size=3, bad_words_ids=banned)
print(tokenizer.decode(dogless_ids))

In [ ]:
# ✅ Check your banned words
assert isinstance(banned, list) and len(banned) == 8, "expected 8 token sequences: 4 words, with and without a leading space"
assert [tokenizer.convert_tokens_to_ids("Ġdog")] in banned, "' dog' (with a space) is missing"


def contains(ids, sequence):
    return any(ids[i : i + len(sequence)] == sequence for i in range(len(ids) - len(sequence) + 1))


assert not any(contains(dogless_ids, sequence) for sequence in banned), "a banned token sequence was generated"
print("Looks good ✅")

**3.4 Is the text now free of dogs?** Read it carefully. Why is banning words with `bad_words_ids` a weak tool, for example for keeping offensive language out of a model's output?

---

*Your answer here:*

---

## 4. Looking inside `generate`

`generate` can return much more than the token ids. With `return_dict_in_generate=True` it returns an object with several fields:

- `sequences`: the token ids, as before,
- `scores` (with `output_scores=True`): one tensor of shape `(batch_size, vocab_size)` **per generated token**, with the scores that the next token was chosen from, *after* the logits processors,
- `logits` (with `output_logits=True`): the same, but the raw logits of the model *before* any processing,
- and optionally the attentions and hidden states of every step (`output_attentions`, `output_hidden_states`).

In [ ]:
prompt = "The capital of Denmark is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=8, return_dict_in_generate=True, output_scores=True)

print(type(out).__name__, list(out.keys()))
print("sequences:", tuple(out.sequences.shape))
print(f"scores: a tuple of {len(out.scores)} tensors of shape {tuple(out.scores[0].shape)}")

**4.1 How sure was the model?** Compute the probability that the model gave to each generated token at the moment it was chosen.

Hints:
- `out.scores[t]` holds the scores for the `t`-th new token. A softmax over the vocabulary turns them into probabilities.
- The `t`-th new token is `out.sequences[0, prompt_length + t]`.

In [ ]:
prompt_length = inputs["input_ids"].shape[1]
new_tokens = out.sequences[0, prompt_length:]

token_probs = []
for t, token_id in enumerate(new_tokens):
    probs = ...  # TODO: the probability distribution over the vocabulary at step t, (vocab_size,)
    token_probs.append(...)  # TODO: the probability of the chosen token, as a float
token_probs = torch.tensor(token_probs)

for token_id, p in zip(new_tokens, token_probs):
    print(f"{tokenizer.decode(token_id)!r:>14}  {p:.3f}")
print(f"\nprobability of the whole continuation: {token_probs.prod():.2e}")

Hugging Face has a helper for exactly this, `compute_transition_scores`. It returns the log-probability of every generated token:

In [ ]:
# ✅ Check your probabilities against compute_transition_scores
log_probs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)[0]
assert token_probs.shape == (8,), "expected one probability per new token"
assert torch.allclose(token_probs, log_probs.exp().cpu(), atol=1e-4), "your probabilities differ from compute_transition_scores"
print("Looks good ✅")

**4.2 Look at the probabilities.** Which tokens was the model sure about and which not? The whole continuation has a tiny probability, even though greedy decoding picked the most likely token at every step. Why is that not a problem in itself, and why does it still mean that the most likely *token* at each step need not lead to the most likely *text*?

---

*Your answer here:*

---

### The scores are *processed* logits

`scores` contains the scores *after* all logits processors. With sampling and top-k (week 6: only the `k` most likely tokens may be sampled), the scores of all other tokens are set to $-\infty$, while the raw `logits` are left untouched.

**4.3 Generate 5 tokens with sampling and `top_k=10`**, returning both the scores and the raw logits, and count at every step how many entries of each are finite (not $-\infty$).

Hint: `torch.isfinite(x).sum()` counts the finite entries of a tensor.

In [ ]:
set_seed(0)
out = ...  # TODO: 5 new tokens, sampling with top_k=10, returning a dictionary with the scores and the logits

finite_scores = ...  # TODO: a list with the number of finite entries of out.scores[t][0] for every step t
finite_logits = ...  # TODO: the same for out.logits

print("finite scores per step:", finite_scores)
print("finite logits per step:", finite_logits)

In [ ]:
# ✅ Check your counts
assert len(finite_scores) == 5 and len(finite_logits) == 5, "expected one count per generated token"
assert all(n == 10 for n in finite_scores), "with top_k=10, exactly 10 scores per step should be finite"
assert all(n == model.config.vocab_size for n in finite_logits), "the raw logits should all be finite"
print("Looks good ✅")

**4.4 Optional: explore the other outputs.** Generate a few tokens with `output_attentions=True` and `output_hidden_states=True` and look at the structure of `out.attentions` and `out.hidden_states`: how many entries are there, and what are their shapes? Can you explain the shapes with what you know about GPT-2 (12 layers, 12 heads, 768 dimensions) and about how `generate` reuses earlier computations? (We come back to the latter, the *KV cache*, in notebook 2.)

In [ ]:
# Your exploration here

## 5. Batched generation

Generating for many prompts one at a time is slow; on a GPU, a batch of prompts takes about as long as a single one. Prompts have different lengths, so they have to be **padded** to the same length, as for classification last week. But for a decoder-only model it matters *on which side* the padding goes.

GPT-2 has no padding token, so we use the EOS token for it, as above. By default the tokenizer pads on the **right**:

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
prompts = ["The capital of Denmark is", "My favourite food in the whole wide world is", "Hello"]

tokenizer.padding_side = "right"
batch = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
print(batch["input_ids"])
print(batch["attention_mask"])

outputs = model.generate(**batch, max_new_tokens=15)
for text in tokenizer.batch_decode(outputs, skip_special_tokens=True):
    print(repr(text))

**5.1 Fix the batch.** Generate the same continuations with padding on the **left** and check that they are the same as when every prompt is generated on its own.

In [ ]:
tokenizer.padding_side = ...  # TODO
batch = ...      # TODO: tokenize the prompts as a padded batch
outputs = ...    # TODO: 15 new tokens for the whole batch

new_ids = outputs[:, batch["input_ids"].shape[1]:]
batched = tokenizer.batch_decode(new_ids, skip_special_tokens=True)
one_by_one = [continue_text(prompt, max_new_tokens=15) for prompt in prompts]
for prompt, a, b in zip(prompts, batched, one_by_one):
    print(f"{prompt}|{a!r}\n{' ' * len(prompt)}|{b!r}\n")

In [ ]:
# ✅ Check your batch
assert tokenizer.padding_side == "left", "pad on the left"
assert batch["input_ids"].shape[0] == 3 and (batch["input_ids"][:, -1] != tokenizer.pad_token_id).all(), "every prompt should end in the last column"
# Padding can change the numbers very slightly, which may (rarely) flip a near-tie late in the text, so compare the first words
for a, b in zip(batched, one_by_one):
    assert a.split()[:5] == b.split()[:5], f"the batched continuation {a!r} differs from {b!r}"
print("Looks good ✅")

**5.2 Why does right padding go wrong for generation**, while it was fine for classifying reviews with DistilBERT last week?

---

*Your answer here:*

---

## 6. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 6.1. Default decoding

You call `model.generate(**inputs, max_new_tokens=50)` on GPT-2 without any other arguments. How is each new token chosen?

A. It is sampled from the full distribution<br>
B. It is the most likely token given the text so far (greedy decoding)<br>
C. Beam search with 5 beams finds the most likely continuation<br>
D. It is sampled from the 50 most likely tokens<br>

**Answer:** 

---

### 6.2. Stopping

Which of the following does **not** stop generation?

A. The model generates the token given as `eos_token_id`<br>
B. `max_new_tokens` tokens have been generated<br>
C. The model generates the same token twice in a row<br>
D. One of the `stop_strings` appears in the output<br>

**Answer:** 

---

### 6.3. Scores

You call `generate` with `do_sample=True, top_k=10, output_scores=True, output_logits=True, return_dict_in_generate=True`. What is the difference between `scores` and `logits`?

A. There is none; both are the raw output of the model<br>
B. `scores` are probabilities, `logits` are log-probabilities<br>
C. `scores` are after the logits processors (here all but 10 entries are $-\infty$), `logits` are the raw outputs of the model<br>
D. `scores` contain one tensor per layer, `logits` one per generated token<br>

**Answer:** 

---

### 6.4. Banned words

You want to stop GPT-2 from writing the word *cat* and pass `bad_words_ids=[tokenizer("cat")["input_ids"]]`. Why can the model still write "My cat is"?

A. `bad_words_ids` only works with beam search<br>
B. Inside a sentence, *cat* is the token `" cat"` with a leading space, which is a different token from `"cat"`<br>
C. `bad_words_ids` only lowers the probability of the token, it does not forbid it<br>
D. GPT-2's tokenizer lowercases all text<br>

**Answer:** 

---

### 6.5. Padding for generation

Why are prompts padded on the **left** when a decoder-only model generates for a batch of prompts?

A. So that the last position of every row is a real token, and the new tokens are appended directly after the prompt<br>
B. Because the attention mask only works for padding on the left<br>
C. Because GPT-2 reads text from right to left<br>
D. Left padding makes the batch shorter<br>

**Answer:** 